# pypdf vs pdfplumber — Day 15

Covers `daily-plans/month-1/day-015.md`'s "pypdf, pdfplumber comparison" task, using
a **real SEC 10-K** (Apple's, downloaded via `month-1/15sec_download.py`) instead of
a toy PDF — so what you see here is exactly what you'll hit when building
Project 1's document-loading pipeline.

**Why this matters at all:** before you can chunk and embed a 10-K (Days 13-14's
work), you first have to get plain text *out* of the PDF. That step sounds trivial
and isn't — PDFs don't actually store "text," they store positioned characters on a
page. Every extraction library is really just guessing, from character positions,
what counts as a word, a line, and a table cell.

In [1]:
import time
from pypdf import PdfReader
import pdfplumber

PDF_PATH = "../tracker/projects/1-sec-10k-analyzer/AAPL_10K_2025-10-31.pdf"
print("Ready. Using:", PDF_PATH)

Ready. Using: ../tracker/projects/1-sec-10k-analyzer/AAPL_10K_2025-10-31.pdf


## 1. pypdf — fast, simple, no table awareness

`pypdf` reads a PDF page by page and gives you back a flat text blob per page.
It's fast and has zero heavy dependencies, which makes it the default choice for
plain prose — like the Risk Factors section.

In [2]:
reader = PdfReader(PDF_PATH)
print("Total pages:", len(reader.pages))

# page 5 is early in the doc -- part of the prose-heavy Risk Factors section
page_text = reader.pages[5].extract_text()
print(page_text[:500])

Total pages: 111
Apple Watch Ultra 3. The Company’s line of wireless headphones includes AirPods, AirPods Pro, AirPods Max
and Beats products. Apple Vision Pro™ is the Company’s spatial computer based on its visionOS operating system.
Home includes Apple TV 4K, the Company’s media streaming and gaming device based on its tvOS operating
system, and HomePod and HomePod mini, high-fidelity wireless smart speakers.
Accessories includes Apple-branded and third-party accessories.
® ® ® ®
® ®
® ®
® ®
Apple Inc. | 2025 


## 2. pdfplumber — slower, but table-aware

`pdfplumber` does more work per page (it reconstructs the actual layout — rows,
columns, spacing) which makes it slower, but far better once tables show up.

In [3]:
with pdfplumber.open(PDF_PATH) as pdf:
    print("Total pages:", len(pdf.pages))
    page_text = pdf.pages[5].extract_text()
    print(page_text[:500])

Total pages: 111
Apple Watch Ultra® 3. The Company’s line of wireless headphones includes AirPods®, AirPods Pro®, AirPods Max®
and Beats® products. Apple Vision Pro™ is the Company’s spatial computer based on its visionOS® operating system.
Home includes Apple TV 4K®, the Company’s media streaming and gaming device based on its tvOS® operating
system, and HomePod® and HomePod mini®, high-fidelity wireless smart speakers.
Accessories includes Apple-branded and third-party accessories.
Apple Inc. | 2025 Form 10-K 


Notice: on plain prose (like this Risk Factors page), the two libraries produce
nearly identical output. The real difference only shows up once a page has an
actual **table** — which is exactly what a 10-K's financial statements are.

## 3. The real test: a financial statement table

Page 54 (0-indexed) of this filing is Apple's `CONSOLIDATED STATEMENTS OF
OPERATIONS` — a genuine income statement with real dollar figures. Same page,
both libraries, so you can see the difference directly instead of taking it on faith.

In [4]:
TABLE_PAGE = 54

print("--- pypdf ---")
print(reader.pages[TABLE_PAGE].extract_text()[:500])

--- pypdf ---
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27,
2025
September 28,
2024
September 30,
2023
Net sales:
   Products $ 307,003$ 294,866$ 298,085
   Services 109,158 96,169 85,200
Total net sales 416,161 391,035 383,285
Cost of sales:
   Products 194,116 185,233 189,282
   Services 26,844 25,119 24,855
Total cost of sales 220,960 210,352 214,137
Gross margin 195,201 180,683 169,


In [5]:
print("--- pdfplumber (plain text mode) ---")
with pdfplumber.open(PDF_PATH) as pdf:
    print(pdf.pages[TABLE_PAGE].extract_text()[:500])

--- pdfplumber (plain text mode) ---
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27, September 28, September 30,
2025 2024 2023
Net sales:
Products $ 307,003 $ 294,866 $ 298,085
Services 109,158 96,169 85,200
Total net sales 416,161 391,035 383,285
Cost of sales:
Products 194,116 185,233 189,282
Services 26,844 25,119 24,855
Total cost of sales 220,960 210,352 214,137
Gross margin 195,201 180,683 169,148
Operat


Look closely at the numbers: pypdf runs `$` signs and dollar figures together with
no space (`307,003$ 294,866`) because it has no concept of "this is a new table
column" — it just concatenates characters left-to-right in reading order.
pdfplumber keeps a real space between columns because it's tracking actual x/y
positions on the page, not just character order.

That's a real problem for RAG: if you later embed `pypdf`'s garbled version, a
question like "what was Apple's 2025 product revenue?" has to match against text
where the number is glued to a stray `$` sign and the *next* year's number —
noisier input, worse retrieval.

## 4. pdfplumber's real advantage: `extract_tables()`

This is the actual reason to reach for pdfplumber on financial statements — it can
give you the table back as **structured rows**, not just better-spaced text.

In [6]:
with pdfplumber.open(PDF_PATH) as pdf:
    table = pdf.pages[TABLE_PAGE].extract_tables()[0]
    for row in table[:10]:
        print(row)

['Products', '$', '307,003', '', '$', '294,866', '', '$', '298,085']
['Services', '109,158', None, '', '96,169', None, '', '85,200', None]
['Total net sales', '416,161', None, '', '391,035', None, '', '383,285', None]
['', '', None, '', '', None, '', '', None]
['Cost of sales:', '', None, '', '', None, '', '', None]
['Products', '194,116', None, '', '185,233', None, '', '189,282', None]
['Services', '26,844', None, '', '25,119', None, '', '24,855', None]
['Total cost of sales', '220,960', None, '', '210,352', None, '', '214,137', None]
['Gross margin', '195,201', None, '', '180,683', None, '', '169,148', None]
['', '', None, '', '', None, '', '', None]


Each row is a real Python list — `['Products', '$', '307,003', ...]` — something
you could load straight into a dict or DataFrame with actual column meaning, instead
of a wall of loosely-spaced text you'd have to regex apart yourself.

## 5. Speed, measured on the full document

pdfplumber's extra layout work has a real cost. Worth knowing before you pick one
for a pipeline that has to process hundreds of pages per filing, times thousands of
filings.

In [7]:
start = time.time()
pypdf_text = "".join(page.extract_text() for page in reader.pages)
pypdf_time = time.time() - start

start = time.time()
with pdfplumber.open(PDF_PATH) as pdf:
    plumber_text = "".join(page.extract_text() or "" for page in pdf.pages)
plumber_time = time.time() - start

print(f"pypdf:      {pypdf_time:.2f}s  ({len(pypdf_text):,} chars)")
print(f"pdfplumber: {plumber_time:.2f}s  ({len(plumber_text):,} chars)")
print(f"pdfplumber was {plumber_time / pypdf_time:.1f}x slower on this document.")

pypdf:      1.49s  (207,963 chars)
pdfplumber: 3.62s  (206,019 chars)
pdfplumber was 2.4x slower on this document.


## 6. Play with it

Things worth trying yourself:
- Change `TABLE_PAGE` to explore other pages (try the balance sheet or cash flow
  statement — search for `"CONSOLIDATED BALANCE SHEET"` in `pypdf_text`).
- Point `PDF_PATH` at `MSFT_10K_2026-07-29.pdf` or `JPM_10K_2026-02-13.pdf`
  (same folder) — JPM's filing is ~800 pages, a good test of the speed gap at
  real scale.
- Try `page.extract_tables()` on a page pypdf couldn't parse cleanly and see how
  much cleaner the recovered numbers are.

In [8]:
# your playground -- try a different page or a different filing


## 7. How this plugs into Project 1

The real pipeline this feeds: **prose sections** (Business, Risk Factors, MD&A) can
use the faster `pypdf` extraction — they're just paragraphs, no tables to lose.
**Financial statement sections** need `pdfplumber`'s `extract_tables()` so numbers
stay attached to their correct row/column labels before they ever get chunked and
embedded (Days 13-14's pipeline). Mixing both, page-type-aware, is the realistic
approach — not picking one library for the entire document.